# PROYECTO: AVANCE 1 - MODELADO Y PRESENTACIÓN DE DATOS
## EDA, Limpieza, Preprocesamiento y Modelado No Supervisado (Clustering)
**Objetivo:** Generar el pipeline analítico completo y exportar los gráficos obligatorios a **300 DPI** en la carpeta `graficos_avance1/` para las diapositivas.

### 1. Carga del Dataset
Asegúrate de asignar tu dataset a la variable `df`.

In [5]:
import pandas as pd

# ------------------------------------------------------------
# CARGA DE DATOS: Actualiza la ruta a tu archivo CSV o Excel
# ------------------------------------------------------------
# Ejemplo: df = pd.read_csv('../datasets/mi_dataset.csv')

try:
    print("Dimensiones actuales del dataset:", df.shape)
    display(df.head())
except NameError:
    # Descomenta y ajusta con el nombre de tu archivo:
    # df = pd.read_csv('tu_dataset.csv')
    print("Por favor asigna tu dataset a la variable 'df' antes de continuar.")

Por favor asigna tu dataset a la variable 'df' antes de continuar.


### 2. Configuración del Entorno, Estilo Visual y Creación de Carpeta

In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster import hierarchy as sch
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Crear carpeta para exportación de gráficos de alta resolución
os.makedirs('graficos_avance1', exist_ok=True)

# Configuración estética profesional institucional
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.labelweight': 'bold',
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'figure.titlesize': 14,
    'figure.dpi': 300
})

# Detección automática de variables numéricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if 'id' not in c.lower()]

print("Carpeta 'graficos_avance1' lista.")
print(f"Variables numéricas detectadas ({len(num_cols)}): {num_cols}")

NameError: name 'df' is not defined

## CRITERIO 2: ANÁLISIS EXPLORATORIO DE DATOS (EDA)
### 2.1 EDA Univariado: Histogramas con KDE y Boxplots Individuales

In [ ]:
cols_eda = num_cols[:6] if len(num_cols) >= 6 else num_cols
n_vars = len(cols_eda)
n_cols_plot = 3 if n_vars >= 3 else n_vars
n_rows_plot = int(np.ceil(n_vars / n_cols_plot))

# 1. Histogramas con KDE
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(5 * n_cols_plot, 3.8 * n_rows_plot))
axes = np.array(axes).reshape(-1)

for idx, col in enumerate(cols_eda):
    sns.histplot(df[col], kde=True, ax=axes[idx], color='#1f77b4', edgecolor='black', alpha=0.6)
    axes[idx].set_title(f'Distribución de {col}', pad=10)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frecuencia')

for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('EDA Univariado: Histogramas y Densidad (KDE)', y=1.02)
plt.tight_layout()
plt.savefig('graficos_avance1/eda_univariado_histogramas.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Boxplots Individuales
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(5 * n_cols_plot, 3.5 * n_rows_plot))
axes = np.array(axes).reshape(-1)

for idx, col in enumerate(cols_eda):
    sns.boxplot(y=df[col], ax=axes[idx], color='#aec7e8', flierprops={'marker':'o', 'markersize':4, 'markerfacecolor':'#d62728'})
    axes[idx].set_title(f'Dispersión y Outliers: {col}', pad=10)
    axes[idx].set_ylabel(col)

for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('EDA Univariado: Diagramas de Caja Iniciales', y=1.02)
plt.tight_layout()
plt.savefig('graficos_avance1/eda_univariado_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n[HALLAZGO PARA DIAPOSITIVA]')
print('1. La mayoría de variables presentan asimetría y colas pronunciadas hacia la derecha.')
print('2. Se constata la presencia de valores atípicos que justifican un tratamiento previo al agrupamiento.')

### 2.2 EDA Multivariado: Matriz de Correlación y Gráfico de Dispersión

In [ ]:
# 1. Matriz de Correlación (Heatmap) con Máscara Triangular
plt.figure(figsize=(9, 7))
corr_matrix = df[num_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.8, cbar_kws={'shrink': 0.8, 'label': 'Coeficiente r de Pearson'}
)
plt.title('Matriz de Correlación Lineal de Pearson', pad=15)
plt.savefig('graficos_avance1/eda_multivariado_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Scatter plot de mayor correlación
corr_unstacked = corr_matrix.abs().unstack()
corr_unstacked = corr_unstacked[corr_unstacked < 0.9999].sort_values(ascending=False)
var_x, var_y = (corr_unstacked.index[0]) if len(corr_unstacked) > 0 else (num_cols[0], num_cols[1])
r_val = corr_matrix.loc[var_x, var_y]

plt.figure(figsize=(8, 5.5))
sns.regplot(
    data=df, x=var_x, y=var_y,
    scatter_kws={'alpha': 0.5, 'color': '#1f77b4', 's': 30},
    line_kws={'color': '#d62728', 'linewidth': 2, 'label': f'Ajuste Lineal (r = {r_val:.2f})'}
)
plt.title(f'Relación Bivariada: {var_x} vs. {var_y}', pad=12)
plt.xlabel(var_x)
plt.ylabel(var_y)
plt.legend(frameon=True)
plt.savefig('graficos_avance1/eda_multivariado_scatterplot.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n[HALLAZGO PARA DIAPOSITIVA]')
print(f'1. Se confirma una correlación lineal de r = {r_val:.2f} entre {var_x} y {var_y}, señalando codependencia.')
print('2. Esta relación fundamenta la aplicación de PCA para condensar varianza redundante en componentes sintéticos.')

## CRITERIO 3: LIMPIEZA Y PREPROCESAMIENTO DE DATOS
### 3.1 Visualización y Tratamiento de Valores Nulos

In [ ]:
# Diagnóstico inicial
null_series = (df.isnull().mean() * 100).sort_values(ascending=True)

plt.figure(figsize=(8, max(4, len(df.columns) * 0.35)))
bars = plt.barh(null_series.index, null_series.values, color='#e6550d', edgecolor='black', alpha=0.8)
plt.title('Porcentaje de Valores Nulos antes de la Limpieza', pad=15)
plt.xlabel('% de Valores Faltantes')
plt.xlim(0, max(15, null_series.max() * 1.15 if null_series.max() > 0 else 10))

for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.3, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center', fontsize=9)

plt.savefig('graficos_avance1/limpieza_nulos_antes.png', dpi=300, bbox_inches='tight')
plt.show()

# Imputación guiada
df_clean = df.copy()
for col in df_clean.select_dtypes(include=[np.number]).columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in df_clean.select_dtypes(include=['object', 'category']).columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

nulos_totales = df_clean.isnull().sum().sum()
assert nulos_totales == 0, 'Persisten nulos.'
print(f'Verificación exitosa: 0 valores nulos restantes en el dataset limpio (df_clean).')

print('\n[HALLAZGO PARA DIAPOSITIVA]')
print('1. La imputación con la mediana protegió la distribución ante el sesgo de outliers sin perder observaciones.')
print('2. Se logró consolidar el 100% de la muestra para el posterior entrenamiento de clustering.')

### 3.2 Tratamiento de Outliers (IQR Capping) y Escalamiento (StandardScaler)

In [ ]:
# 1. Tratamiento IQR Capping
df_no_outliers = df_clean.copy()
for col in num_cols:
    Q1 = df_no_outliers[col].quantile(0.25)
    Q3 = df_no_outliers[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    df_no_outliers[col] = np.clip(df_no_outliers[col], lim_inf, lim_sup)

# Boxplots comparativos 'Antes vs Después'
cols_comp = num_cols[:4] if len(num_cols) >= 4 else num_cols
fig, axes = plt.subplots(len(cols_comp), 2, figsize=(10, 3 * len(cols_comp)))
if len(cols_comp) == 1:
    axes = np.array([axes])

for i, col in enumerate(cols_comp):
    sns.boxplot(x=df_clean[col], ax=axes[i, 0], color='#fc9272')
    axes[i, 0].set_title(f'{col} (Original con Outliers)')
    axes[i, 0].set_xlabel('')
    
    sns.boxplot(x=df_no_outliers[col], ax=axes[i, 1], color='#a1d99b')
    axes[i, 1].set_title(f'{col} (Post Capping IQR)')
    axes[i, 1].set_xlabel('')

plt.suptitle('Control de Outliers: Comparativa Antes vs. Después (Método IQR)', y=1.01)
plt.tight_layout()
plt.savefig('graficos_avance1/tratamiento_outliers_comparativo.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Escalamiento con StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_no_outliers[num_cols])
df_scaled = pd.DataFrame(X_scaled, columns=num_cols)

plt.figure(figsize=(9, 5))
for col in cols_comp:
    sns.kdeplot(df_scaled[col], label=col, linewidth=2)

plt.axvline(0, color='gray', linestyle='--', alpha=0.7, label='Media centrada (0.0)')
plt.title('Distribución Post-Escalamiento (StandardScaler: mu=0, sigma=1)', pad=12)
plt.xlabel('Desviaciones Estándar (Z-Score)')
plt.ylabel('Densidad')
plt.legend(frameon=True)
plt.savefig('graficos_avance1/escalamiento_distribuciones.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n[HALLAZGO PARA DIAPOSITIVA]')
print('1. El acotamiento IQR mitigó valores extremos sin eliminar filas indispensables para el negocio.')
print('2. La estandarización puso todas las variables en igualdad de peso ante el cálculo de distancias euclidianas.')

## CRITERIO 4: MODELADO NO SUPERVISADO (CLUSTERING K-MEANS & JERÁRQUICO)
### 4.1 Determinación de K Óptimo: Método del Codo y Coeficiente de Silueta

In [ ]:
K_range = range(2, 11)
inercia = []
siluetas = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inercia.append(km.inertia_)
    siluetas.append(silhouette_score(X_scaled, labels))

k_optimo = K_range[np.argmax(siluetas)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico del Codo
ax1.plot(K_range, inercia, marker='o', linewidth=2, color='#1f77b4', markersize=7)
ax1.axvline(k_optimo, color='#d62728', linestyle='--', linewidth=2, label=f'K óptimo sugerido = {k_optimo}')
ax1.set_title('Método del Codo (Inercia vs. K)', pad=10)
ax1.set_xlabel('Número de Clústeres (K)')
ax1.set_ylabel('Inercia WCSS')
ax1.set_xticks(list(K_range))
ax1.legend(frameon=True)

# Gráfico de Silueta
ax2.plot(K_range, siluetas, marker='s', linewidth=2, color='#2ca02c', markersize=7)
ax2.axvline(k_optimo, color='#d62728', linestyle='--', linewidth=2, label=f'K óptimo sugerido = {k_optimo}')
ax2.set_title('Coeficiente de Silueta Promedio vs. K', pad=10)
ax2.set_xlabel('Número de Clústeres (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(list(K_range))
ax2.legend(frameon=True)

plt.suptitle('Evaluación de Hiperparámetro K para K-Means', y=1.03)
plt.tight_layout()
plt.savefig('graficos_avance1/clustering_metodo_codo_silueta.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'K óptimo seleccionado según Silueta: {k_optimo} (Score: {max(siluetas):.3f})')
print('\n[HALLAZGO PARA DIAPOSITIVA]')
print(f'1. El punto de inflexión del codo y el pico de silueta concuerdan en K = {k_optimo}.')
print('2. Esta configuración minimiza la dispersión interna asegurando grupos nítidamente diferenciados.')

### 4.2 Agrupamiento Jerárquico: Dendrograma con Método Ward

In [ ]:
sample_size = min(300, len(X_scaled))
np.random.seed(42)
sample_indices = np.random.choice(len(X_scaled), size=sample_size, replace=False)
X_sample = X_scaled[sample_indices]

linkage_matrix = sch.linkage(X_sample, method='ward', metric='euclidean')
corte_distancia = linkage_matrix[-(k_optimo-1), 2]

plt.figure(figsize=(12, 6))
sch.dendrogram(
    linkage_matrix, no_labels=True,
    color_threshold=corte_distancia
)
plt.axhline(corte_distancia, color='#d62728', linestyle='--', linewidth=2, label=f'Línea de Corte (K = {k_optimo})')
plt.title(f'Dendrograma de Agrupamiento Jerárquico (Método Ward) - n={sample_size}', pad=15)
plt.xlabel('Observaciones')
plt.ylabel('Distancia Euclidiana de Enlace')
plt.legend(loc='upper right', frameon=True)
plt.savefig('graficos_avance1/clustering_dendrograma.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n[HALLAZGO PARA DIAPOSITIVA]')
print('1. El dendrograma muestra la agrupación jerárquica natural de observaciones con mínima varianza interna.')
print(f'2. La altura del corte horizontal confirma una división natural en {k_optimo} conglomerados principales.')

### 4.3 Resultados Finales: Dispersión 2D con PCA y Tabla Resumen de Negocio

In [ ]:
# 1. Modelo K-Means final
kmeans_final = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
df_no_outliers['cluster'] = kmeans_final.fit_predict(X_scaled)
df_clean['cluster'] = df_no_outliers['cluster']

# 2. Reducción PCA a 2 componentes
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
centroides_pca = pca.transform(kmeans_final.cluster_centers_)
var_exp = pca.explained_variance_ratio_ * 100

df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['cluster'] = df_clean['cluster'].astype(str)

# Gráfico 2D con Centroides
plt.figure(figsize=(9, 6.5))
sns.scatterplot(
    data=df_pca, x='PC1', y='PC2', hue='cluster',
    palette=sns.color_palette('tab10', n_colors=k_optimo), alpha=0.6, s=40, edgecolor='none'
)
plt.scatter(
    centroides_pca[:, 0], centroides_pca[:, 1],
    s=250, c='#d62728', marker='X', edgecolors='black', linewidth=1.5,
    label='Centroides de Clúster'
)
plt.title(f'Segmentación 2D con PCA (Varianza Explicada: {sum(var_exp):.1f}%)', pad=12)
plt.xlabel(f'Componente Principal 1 ({var_exp[0]:.1f}%)')
plt.ylabel(f'Componente Principal 2 ({var_exp[1]:.1f}%)')
plt.legend(title='Clúster', frameon=True, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.savefig('graficos_avance1/clustering_segmentacion_pca_2d.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Tabla Resumen y Perfilamiento
resumen_cluster = df_no_outliers.groupby('cluster')[num_cols].mean().round(2)
conteo_cluster = df_no_outliers['cluster'].value_counts().rename('Registros')
pct_cluster = (df_no_outliers['cluster'].value_counts(normalize=True) * 100).round(1).rename('%_Total')
tabla_perfil = pd.concat([conteo_cluster, pct_cluster, resumen_cluster], axis=1)

fig, ax = plt.subplots(figsize=(max(8, len(tabla_perfil.columns) * 1.3), 1.2 + 0.6 * len(tabla_perfil)))
ax.axis('off')
ax.axis('tight')
tabla_plot = ax.table(
    cellText=tabla_perfil.values,
    colLabels=tabla_perfil.columns,
    rowLabels=[f'Clúster {i}' for i in tabla_perfil.index],
    cellLoc='center', loc='center'
)
tabla_plot.auto_set_font_size(False)
tabla_plot.set_fontsize(9)
tabla_plot.scale(1.2, 1.6)

for (row, col), cell in tabla_plot.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1f77b4')
        cell.set_text_props(color='white', weight='bold')
    elif col == -1:
        cell.set_facecolor('#e9ecef')
        cell.set_text_props(weight='bold')

plt.title('Perfilamiento y Métricas Promedio por Clúster de Negocio', pad=20, weight='bold')
plt.savefig('graficos_avance1/clustering_tabla_perfilamiento.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n[HALLAZGO PARA DIAPOSITIVA]')
print(f'1. El espacio bidimensional retiene el {sum(var_exp):.1f}% de la variabilidad, mostrando agrupaciones definidas.')
print('2. La tabla de perfilamiento traduce los centroides en segmentos claros para la toma de decisiones estratégicas.')

### 5. Verificación de Gráficos Guardados en Alta Resolución (300 DPI)

In [ ]:
import glob
graficos_generados = sorted(glob.glob('graficos_avance1/*.png'))
print(f'Total de gráficos listos para diapositivas: {len(graficos_generados)}\n')
for g in graficos_generados:
    print(f' [OK] -> {g}')